In [37]:
print "Om"

SyntaxError: Missing parentheses in call to 'print'. Did you mean print(...)? (730924595.py, line 1)

In [ ]:
print("Om")

Om


In [ ]:
import json
x = '{name:Mani, City:Glen Rock, State: NJ}'
y=json.loads(x)
print(y["City"])

JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)

In [ ]:
import json
x = '{"Name":"Mani", "City":"Glen Rock", "State": "NJ"}'
y=json.loads(x)
print(y["City"])

Glen Rock


In [ ]:
import sys
from typing import Final

class RecordWithDict:
    def __init__(self, record_id: int, symbol: str, price: float) -> None:
        self.record_id = record_id
        self.symbol = symbol
        self.price = price

class RecordWithSlots:
    __slots__: Final[tuple[str, ...]] = ("record_id", "symbol", "price")

    def __init__(self, record_id: int, symbol: str, price: float) -> None:
        self.record_id = record_id
        self.symbol = symbol
        self.price = price

# Memory footprint comparison
dict_instance = RecordWithDict(101, "AAPL", 185.50)
slots_instance = RecordWithSlots(101, "AAPL", 185.50)

# __dict__ adds substantial overhead per object
overhead_mem = sys.getsizeof(dict_instance.__dict__)
dict_mem = sys.getsizeof(dict_instance) + sys.getsizeof(dict_instance.__dict__)
slots_mem = sys.getsizeof(slots_instance)
print(f"overhead_mem total memory: {overhead_mem} bytes")
print(f"RecordWithDict total memory: {dict_mem} bytes")
print(f"RecordWithSlots total memory: {slots_mem} bytes")

overhead_mem total memory: 296 bytes
RecordWithDict total memory: 344 bytes
RecordWithSlots total memory: 56 bytes


In [ ]:
class Base:
    __slots__ = ("id",)

class ChildUnslotted(Base):
    pass

class ChildSlotted(Base):
    __slots__ = ("timestamp",)

c_unslotted = ChildUnslotted()
c_slotted = ChildSlotted()

print(hasattr(c_unslotted, "__dict__"))  # True (Dict overhead reintroduced)
print(hasattr(c_slotted, "__dict__"))    # False (Maintains compact memory layout)

True
False


In [ ]:
from typing import Final

class DataPartition:
    __slots__: Final[tuple[str, ...]] = ("partition_id", "source_table", "is_loaded")

    partition_id: int
    source_table: str
    is_loaded: bool

    def __init__(self, partition_id: int, source_table: str) -> None:
        self.partition_id = partition_id
        self.source_table = source_table
        self.is_loaded = False

    def load(self) -> None:
        self.is_loaded = True


    def summary(self) -> str:
        status: str = "LOADED" if self.is_loaded else "PENDING"
        return f"Partition {self.partition_id} from {self.source_table}: {status}"

    def __repr__(self) -> str:
        return(
            f"DataPartition(partition_id={self.partition_id}," 
            f"source_table={self.source_table!r},"
            f"is_loaded={self.is_loaded})"
            )
    
    def __eq__(self, other: object) -> bool:
        if not isinstance(other, DataPartition):
            return NotImplemented
        return (
            self.partition_id == other.partition_id and
            self.source_table == other.source_table 
                )
    
    
    def __hash__(self) -> int:
         return hash((self.partition_id, self.source_table))
        
print ("Creating DataPartition instance...")
partition1 = DataPartition(1, "users")
print(partition1.summary())  # Output: Partition 1 from users: PENDING
partition2 = DataPartition(1, "users")
partition2.load()
print(partition2.summary())  # Output: Partition 1 from users: LOADED
print (partition1)
print (partition2)
print(partition1 == partition2 )  # Output: True (same partition_id and source_table)
myset = {partition1, partition2}

# 1. Structural equality works across state transitions
assert partition1 == partition2, "Partitions with same id and source_table must be equal"

# 2. Hash consistency remains invariant
assert hash(partition1) == hash(partition2), "Hashes of equal objects must match"

# 3. Deduplication in sets succeeds
partition_set = {partition1, partition2}
assert len(partition_set) == 1, "Set must deduplicate identical partitions"
print(f"Set contents: {partition_set}")

print(myset)    # Output: {DataPartition(partition_id=1, source_table




Creating DataPartition instance...
Partition 1 from users: PENDING
Partition 1 from users: LOADED
DataPartition(partition_id=1,source_table='users',is_loaded=False)
DataPartition(partition_id=1,source_table='users',is_loaded=True)
True
Set contents: {DataPartition(partition_id=1,source_table='users',is_loaded=False)}
{DataPartition(partition_id=1,source_table='users',is_loaded=False)}


In [ ]:
# Snippet A (Interactive REPL line-by-line)
>>> a = 257
>>> b = 257
>>> a is b


False

In [ ]:
import sys
import time
from typing import Final

# Simulate 5,000,000 incoming ticker records from a broker stream
ITERATIONS: Final[int] = 5_000_000
raw_incoming: list[str] = [f"TICKER_{i % 50}" for i in range(ITERATIONS)]
target: str = "TICKER_25"

# --- Scenario 1: Standard Non-Interned Equality Check ---
t0 = time.perf_counter()
matches_eq: int = 0
for sym in raw_incoming:
    if sym == target:  # Executes PyUnicode_Compare / memcmp
        matches_eq += 1
dur_eq = time.perf_counter() - t0

# --- Scenario 2: Canonical Interned Pointer Check ---
# Interning occurs once at the gateway/deserialization boundary
interned_target: str = sys.intern(target)
interned_incoming: list[str] = [sys.intern(s) for s in raw_incoming]

t1 = time.perf_counter()
matches_is: int = 0
for sym in interned_incoming:
    if sym is interned_target:  # Single pointer compare (CMP)
        matches_is += 1
dur_is = time.perf_counter() - t1

print(f"Value Equality (`==`): {dur_eq:.4f}s")
print(f"Pointer Check  (`is`): {dur_is:.4f}s ({((dur_eq - dur_is) / dur_eq) * 100:.1f}% faster)")

Value Equality (`==`): 0.2738s
Pointer Check  (`is`): 0.2488s (9.1% faster)


In [38]:
def max_volume(height: list[int]) -> int:
    if not height:
        return 0
    left = 0
    right = len(height) - 1
    max_area = 0
    while left < right:
        current_area = min(height[left], height[right]) * (right - left)
        max_area = max(max_area, current_area)
        if height[left] < height[right]:
            left += 1
        else:
            right -= 1
    return max_area

max_volume([1,8,6,2,5,4,8,3,7])  # Output: 49


49

In [43]:
class Solution:
    def validPalindrome(self, s: str) -> bool:
       
        def is_valid_palindrome_rage(left:int,right:int)->bool:
            while left < right:
                if s[left]!=s[right]:
                    return False
                left+=1
                right-=1
            return True

        left:int=0
        right:int= len(s)-1
        while left < right:
          if s[left] !=s[right]:
            return  is_valid_palindrome_rage(left+1,right) or is_valid_palindrome_rage(left,right-1)                    
          left+=1
          right-=1
        return True


a = Solution();
print(a.validPalindrome("abca")) # Output: True
print(a.validPalindrome("abc")) # Output: False

True
False


In [1]:
from decimal import Decimal
from typing import Final

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    StringType,
    StructField,
    StructType,
)


def score_card_transactions(df: DataFrame) -> DataFrame:
    """Classifies risk tiers natively using Catalyst expressions.

    Preserves WholeStageCodegen by compiling branching logic directly into JVM
    bytecode, avoiding PySpark worker daemon allocation and SerDe overhead.
    """
    critical_condition: Final = (F.col("amount") > 10000.0) | (
        F.col("is_foreign") & (~F.col("country").isin(["US", "CA"]))
    )
    high_condition: Final = F.col("amount") > 5000.0
    medium_condition: Final = (F.col("amount") > 1000.0) & F.col("is_foreign")

    return df.withColumn(
        "risk_category",
        F.when(F.col("amount").isNull(), F.lit("UNKNOWN"))
        .when(critical_condition, F.lit("CRITICAL"))
        .when(high_condition, F.lit("HIGH"))
        .when(medium_condition, F.lit("MEDIUM"))
        .otherwise(F.lit("LOW")),
    )


def build_mock_dataset(spark: SparkSession) -> DataFrame:
    """Constructs edge cases targeting all evaluation branches."""
    schema = StructType(
        [
            StructField("tx_id", StringType(), False),
            StructField("amount", DoubleType(), True),
            StructField("country", StringType(), False),
            StructField("is_foreign", BooleanType(), False),
        ]
    )

    data = [
        ("TX-001", None, "US", False),  # Branch 1: Null check -> UNKNOWN
        (
            "TX-002",
            12500.00,
            "US",
            False,
        ),  # Branch 2a: Amount > 10k -> CRITICAL
        (
            "TX-003",
            250.00,
            "GB",
            True,
        ),  # Branch 2b: Foreign & country != US/CA -> CRITICAL
        (
            "TX-004",
            7500.00,
            "CA",
            True,
        ),  # Branch 3: Amount > 5k (Foreign CA) -> HIGH
        ("TX-005", 6000.00, "US", False),  # Branch 3: Amount > 5k -> HIGH
        ("TX-006", 2500.00, "CA", True),  # Branch 4: Amount > 1k & Foreign -> MEDIUM
        ("TX-007", 2500.00, "US", False),  # Branch 5: Default fallback -> LOW
        ("TX-008", 45.50, "US", False),  # Branch 5: Default fallback -> LOW
    ]

    return spark.createDataFrame(data=data, schema=schema)


def main() -> None:
    spark: Final[SparkSession] = (
        SparkSession.builder.master("local[2]")
        .appName("RiskScoringVerification")
        .config("spark.ui.enabled", "false")
        .config("spark.sql.shuffle.partitions", "2")
        .getOrCreate()
    )

    # 1. Ingestion of mock data
    raw_df = build_mock_dataset(spark)

    # 2. Execution of pure Catalyst transformation
    scored_df = score_card_transactions(raw_df)

    # 3. Plan Inspection
    print("\n" + "=" * 80)
    print("CATALYST FORMATTED PHYSICAL PLAN")
    print("=" * 80)
    scored_df.explain(mode="formatted")

    # 4. Data Inspection
    print("\n" + "=" * 80)
    print("EVALUATED OUTPUT DATASET")
    print("=" * 80)
    scored_df.show(truncate=False)

    spark.stop()


if __name__ == "__main__":
    main()


CATALYST FORMATTED PHYSICAL PLAN
== Physical Plan ==
* Project (2)
+- * Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [4]: [tx_id#0, amount#1, country#2, is_foreign#3]
Arguments: [tx_id#0, amount#1, country#2, is_foreign#3], MapPartitionsRDD[4] at applySchemaToPythonRDD at DirectMethodHandleAccessor.java:104, ExistingRDD, UnknownPartitioning(0)

(2) Project [codegen id : 1]
Output [5]: [tx_id#0, amount#1, country#2, is_foreign#3, CASE WHEN isnull(amount#1) THEN UNKNOWN WHEN ((amount#1 > 10000.0) OR (is_foreign#3 AND NOT country#2 IN (US,CA))) THEN CRITICAL WHEN (amount#1 > 5000.0) THEN HIGH WHEN ((amount#1 > 1000.0) AND is_foreign#3) THEN MEDIUM ELSE LOW END AS risk_category#4]
Input [4]: [tx_id#0, amount#1, country#2, is_foreign#3]



EVALUATED OUTPUT DATASET


Py4JJavaError: An error occurred while calling o97.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, most recent failure: Lost task 0.0 in stage 0.0 (TID 0) (DESKTOP-ICE3CGJ executor driver): org.apache.spark.SparkException: Python worker failed to connect back.
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:328)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:179)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:326)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:637)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:73)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:374)
	at org.apache.spark.rdd.RDD.iterator(RDD.scala:338)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRDD.scala:57)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoin